# Banca y fraude: monto de la transacción anterior por cliente

Tipo: SQL · Nivel: 1 · Apoyo: Guiado · Tema: SQL - window functions · Etapa: exploración · Tiempo objetivo: 20-30 min (leer la teoría y el paso a paso NO cuenta en Tiempo Real) :a

In [1]:
import sqlite3
import pandas as pd

print('Setup Okay')

Setup Okay


In [2]:
con = sqlite3.connect(":memory:")
con.executescript("""
-- pega aquí los CREATE TABLE e INSERT del reto

CREATE TABLE transacciones (
    id INTEGER PRIMARY KEY,
    cliente_id VARCHAR(10) NOT NULL,
    fecha DATE NOT NULL,
    monto NUMERIC(10,2) NOT NULL
);

INSERT INTO transacciones (id, cliente_id, fecha, monto) VALUES
(1, 'C001', '2026-08-01', 150.00),
(2, 'C001', '2026-08-05', 200.00),
(3, 'C001', '2026-08-12', 5000.00),
(4, 'C002', '2026-08-02', 80.00),
(5, 'C002', '2026-08-09', 95.00),
(6, 'C002', '2026-08-20', 120.00),
(7, 'C003', '2026-08-03', 300.00),
(8, 'C003', '2026-08-04', 310.00),
(9, 'C003', '2026-08-06', 295.00),
(10, 'C004', '2026-08-01', 60.00),
(11, 'C004', '2026-08-15', 4000.00),
(12, 'C004', '2026-08-16', 70.00);

""")

In [6]:
df = pd.read_sql_query("SELECT * FROM transacciones", con)
df.style.format({'monto': '{:.2f}'})
df


,id,cliente_id,fecha,monto
0,1,C001,2026-08-01,150
1,2,C001,2026-08-05,200
2,3,C001,2026-08-12,5000
3,4,C002,2026-08-02,80
4,5,C002,2026-08-09,95
5,6,C002,2026-08-20,120
6,7,C003,2026-08-03,300
7,8,C003,2026-08-04,310
8,9,C003,2026-08-06,295
9,10,C004,2026-08-01,60


In [13]:
df = pd.read_sql("""

-- Seleccion de Features

SELECT
    cliente_id,
    fecha,
    monto,

    -- Utilizar la window function LAG para obtener el monto de la transacción anterior
    LAG(monto, 1) OVER (PARTITION BY cliente_id ORDER BY fecha) as monto_anterior,

    -- Operacion Aritmetica para calcular la diferencia
    ROUND(
        monto - LAG(monto, 1) OVER (PARTITION BY cliente_id ORDER BY fecha),
        2
    ) AS diferencia

FROM transacciones
ORDER BY cliente_id, fecha;
""", con)

df

,cliente_id,fecha,monto,monto_anterior,diferencia
0,C001,2026-08-01,150,NaN,NaN
1,C001,2026-08-05,200,150.0,50.0
2,C001,2026-08-12,5000,200.0,4800.0
3,C002,2026-08-02,80,NaN,NaN
4,C002,2026-08-09,95,80.0,15.0
5,C002,2026-08-20,120,95.0,25.0
6,C003,2026-08-03,300,NaN,NaN
7,C003,2026-08-04,310,300.0,10.0
8,C003,2026-08-06,295,310.0,-15.0
9,C004,2026-08-01,60,NaN,NaN


# Observaciones

Una vez calcula la diferencia de transcciones de los clientes, hay ciertas transcciones que merecen ser revisadas como:
1. Cliente ('COO1'): Quien tuvo una diferencia de 4,800.00 entre sus transcacciones
2. Cliente ('C004'): Quien tuvo una diferencia de 3940.00 entre sus transacciones